# 🎭 The Jokesters — Custom WebLLM Model Compiler (Colab)

**Goal:** Produce custom `.wasm` model libraries, quantized weight shards, and `mlc-chat-config.json` artifacts tuned for *The Jokesters* — a multi-agent comedy improv show running entirely in the browser with WebGPU.

When the Comedian lands a punchline before the spotlight dims, that's what we're optimizing for.

---

## What this notebook produces

| Artifact | Purpose |
|----------|---------|
| `*.wasm` (`model_lib`) | TVM-compiled WebGPU runtime with **baked-in** context/prefill memory plan |
| `params_shard_*.bin` + `tensor-cache.json` | Quantized weights |
| `mlc-chat-config.json` | Model + conv template + context knobs |
| `jokesters-webllm-artifacts.zip` | One-click download bundle |

## Why custom compile?

The Jokesters already uses runtime overrides (`context_window_size`, `sliding_window_size`) in `src/config/models.ts`, but a **custom `.wasm`** bakes the tighter memory plan into TVM at compile time — typically **~300 MB less peak VRAM** vs loading a generic 4K-context `.wasm` and overriding to 512.

That difference can be the line between OOM and a smooth autonomous improv scene.

---

> ⚠️ **Colab reality check**
>
> - **Weight conversion + config** — works great on Colab GPU runtime (~20–40 min for 7B).
> - **Full `mlc_llm compile --device webgpu`** — needs Emscripten + MLC-LLM source build (~30–60 min first time). Colab sessions may **timeout** on 8B compiles.
> - **Recommended workflow:** Run conversion + `gen_config` here, download the zip, then compile locally with `scripts/build-vicuna-wasm.sh` or on a machine with the full toolchain.
>
> The Scientist recommends measuring before heroics. The Comedian recommends coffee.


## 0. Runtime checklist

1. **Runtime → Change runtime type → GPU** (T4/L4 is fine for conversion; compile is mostly CPU-bound).
2. If you hit OOM during weight conversion, restart runtime and pick a smaller model (3B) or `q4f32_1`.
3. For gated models (Llama 3.1), set your HuggingFace token in the next cell.


In [ ]:
# @title 0️⃣ Configuration — pick your fighter
# @markdown ### Model & quantization
MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"  # @param ["meta-llama/Meta-Llama-3.1-8B-Instruct", "lmsys/vicuna-7b-v1.5", "meta-llama/Llama-3.2-3B-Instruct"] {allow-input: true}
QUANTIZATION = "q4f16_1"  # @param ["q4f16_1", "q4f32_1"]
# @markdown `q4f16_1` = faster (needs shader-f16). `q4f32_1` = universal WebGPU compat (ultra-low VRAM path).

# @markdown ### Jokesters performance preset
VRAM_PRESET = "standard"  # @param ["standard", "ultra_low"]
# @markdown - **standard**: 1024 ctx — good balance for prerendered multi-agent turns
# @markdown - **ultra_low**: 512 ctx + sliding window — fits tighter GPUs (<4 GB)

# @markdown ### HuggingFace (required for gated Llama models)
HF_TOKEN = ""  # @param {type:"string"}
# @markdown Leave blank for public models like Vicuna.

# @markdown ### Build options
SKIP_TOOLCHAIN_BUILD = False  # @param {type:"boolean"}
# @markdown Skip Emscripten/MLC source build if already done this session.
TRY_COLAB_COMPILE = True  # @param {type:"boolean"}
# @markdown Attempt WASM compile in Colab (may timeout — see fallback section).

import os
from pathlib import Path

MODEL_SHORT = MODEL_ID.split("/")[-1]
WORK_ROOT = Path("/content/jokesters-mlc")
DIST_ROOT = WORK_ROOT / "dist"
HF_CACHE = WORK_ROOT / "hf_models" / MODEL_SHORT
OUTPUT_DIR = DIST_ROOT / f"{MODEL_SHORT}-{QUANTIZATION}-MLC"

PRESETS = {
    "standard": {
        "context_window_size": 1024,
        "prefill_chunk_size": 512,
        "sliding_window_size": 256,
        "attention_sink_size": 4,
        "note": "Balanced for prerendered improv — enough context for 6–8 turns, smaller prefill chunks reduce perceived latency.",
    },
    "ultra_low": {
        "context_window_size": 512,
        "prefill_chunk_size": 512,
        "sliding_window_size": 256,
        "attention_sink_size": 4,
        "note": "Ultra-low VRAM — matches VPS_VICUNA_7B_ULTRA_LOW in src/config/models.ts.",
    },
}
PRESET = PRESETS[VRAM_PRESET]

CONV_TEMPLATES = {
    "meta-llama/Meta-Llama-3.1-8B-Instruct": "llama-3",
    "meta-llama/Llama-3.2-3B-Instruct": "llama-3",
    "lmsys/vicuna-7b-v1.5": "vicuna_v1.1",
}
CONV_TEMPLATE = CONV_TEMPLATES.get(MODEL_ID, "llama-3")

for p in [WORK_ROOT, DIST_ROOT, HF_CACHE.parent, OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

print("🎤 Jokesters compile configuration")
print(f"  Model:        {MODEL_ID}")
print(f"  Quantization: {QUANTIZATION}")
print(f"  Conv template:{CONV_TEMPLATE}")
print(f"  VRAM preset:  {VRAM_PRESET} → ctx={PRESET['context_window_size']}, prefill={PRESET['prefill_chunk_size']}")
print(f"  Output dir:   {OUTPUT_DIR}")
print(f"  Preset note:  {PRESET['note']}")


## 1. Environment setup

Installs `mlc-llm` nightly, system deps, Emscripten, Rust, and clones MLC-LLM source.

> ⏱️ First run: **30–60 minutes**. Subsequent runs in the same session are faster if `SKIP_TOOLCHAIN_BUILD=True`.


In [ ]:
%%capture setup_log
# @title 1️⃣ Install Python packages & system deps
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--pre", "-U",
    "-f", "https://mlc.ai/wheels",
    "mlc-llm-nightly-cu128", "mlc-ai-nightly-cu128",
    "huggingface_hub", "ipywidgets",
])
subprocess.check_call(["apt-get", "update", "-qq"])
subprocess.check_call(["apt-get", "install", "-y", "-qq", "git-lfs", "cmake", "ninja-build", "curl"])
subprocess.check_call(["git", "lfs", "install"])
print("✅ Python packages installed")


In [ ]:
%%capture toolchain_log
# @title 2️⃣ Build toolchain (Emscripten + Rust + MLC Web Runtime)
import os, subprocess, sys
from pathlib import Path

SKIP = SKIP_TOOLCHAIN_BUILD
EMSDK = Path("/content/emsdk")
MLC_SRC = Path("/content/mlc-llm")
os.environ["EMSDK"] = str(EMSDK)

def run(cmd, **kw):
    print("$", " ".join(cmd) if isinstance(cmd, list) else cmd)
    subprocess.check_call(cmd, shell=isinstance(cmd, str), **kw)

if not SKIP:
    # Rust
    if not Path.home().joinpath(".cargo/bin/rustc").exists():
        run("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y", shell=True)
    os.environ["PATH"] = f"{Path.home()}/.cargo/bin:" + os.environ.get("PATH", "")
    run(["rustup", "target", "add", "wasm32-unknown-emscripten"])

    # Emscripten
    if not EMSDK.exists():
        run(["git", "clone", "https://github.com/emscripten-core/emsdk.git", str(EMSDK)])
    run([str(EMSDK / "emsdk"), "install", "tot"])
    run([str(EMSDK / "emsdk"), "activate", "tot"])

    # MLC-LLM source
    if not MLC_SRC.exists():
        run(["git", "clone", "--recursive", "https://github.com/mlc-ai/mlc-llm.git", str(MLC_SRC)])

    # Build web runtime
    env = os.environ.copy()
    env["TVM_HOME"] = str(MLC_SRC / "3rdparty/tvm")
    env["TVM_LIBRARY_PATH"] = str(MLC_SRC / "web/dist/wasm")
    run(f"source {EMSDK}/emsdk_env.sh && cd {MLC_SRC} && ./web/prep_emcc_deps.sh", shell=True, env=env)
    wasm_build = MLC_SRC / "build/wasm"
    wasm_build.mkdir(parents=True, exist_ok=True)
    run(
        f"source {EMSDK}/emsdk_env.sh && cd {wasm_build} && "
        f"emcmake cmake ../.. -DCMAKE_BUILD_TYPE=Release -DUSE_WEBGPU=ON -DUSE_WASM=ON -DCMAKE_CXX_FLAGS='-O3' && "
        f"make -j$(nproc)",
        shell=True,
        env=env,
    )
    print("✅ Toolchain build complete")
else:
    print("⏭️ Skipping toolchain build (SKIP_TOOLCHAIN_BUILD=True)")


## 2. Download base model

Clones the HuggingFace weights. For gated models, ensure `HF_TOKEN` is set above.


In [ ]:
# @title 3️⃣ Download HuggingFace weights
import os, subprocess
from huggingface_hub import snapshot_download

if HF_CACHE.exists() and any(HF_CACHE.iterdir()):
    print(f"✅ Model cache exists: {HF_CACHE}")
else:
    print(f"⬇️ Downloading {MODEL_ID} …")
    token = HF_TOKEN or None
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=str(HF_CACHE),
        local_dir_use_symlinks=False,
        token=token,
    )
    print(f"✅ Downloaded to {HF_CACHE}")

!ls -lh {HF_CACHE} | head -20


## 3. Convert weights & generate config

`convert_weight` quantizes the model. `gen_config` creates the chat template + metadata.

Then we **patch** the config with Jokesters-optimized context/prefill/sliding-window values.

### Why these knobs matter for improv

| Setting | Jokesters impact |
|---------|------------------|
| `context_window_size` | Caps KV cache at load time — biggest VRAM lever |
| `prefill_chunk_size` | Smaller chunks = faster time-to-first-token during prerender |
| `sliding_window_size` | Rolling attention keeps recent banter, drops ancient setup lines |
| `attention_sink_size` | Preserves system prompt + persona anchors (StreamingLLM trick) |


In [ ]:
%%capture convert_log
# @title 4️⃣ convert_weight
import subprocess, sys
cmd = [
    sys.executable, "-m", "mlc_llm", "convert_weight", str(HF_CACHE) + "/",
    "--quantization", QUANTIZATION,
    "-o", str(OUTPUT_DIR),
]
print("Running:", " ".join(cmd))
subprocess.check_call(cmd)
print("✅ Weight conversion complete")


In [ ]:
%%capture genconfig_log
# @title 5️⃣ gen_config + Jokesters preset patch
import json, subprocess, sys

cmd = [
    sys.executable, "-m", "mlc_llm", "gen_config", str(HF_CACHE) + "/",
    "--quantization", QUANTIZATION,
    "--conv-template", CONV_TEMPLATE,
    "-o", str(OUTPUT_DIR),
]
print("Running:", " ".join(cmd))
subprocess.check_call(cmd)

config_path = OUTPUT_DIR / "mlc-chat-config.json"
with open(config_path) as f:
    cfg = json.load(f)

# Patch top-level and nested model_config keys
for key in ["context_window_size", "prefill_chunk_size", "sliding_window_size", "attention_sink_size"]:
    if key in PRESET:
        cfg[key] = PRESET[key]
        if "model_config" in cfg and isinstance(cfg["model_config"], dict):
            cfg["model_config"][key] = PRESET[key]

# Tag for Jokesters traceability
cfg.setdefault("jokesters_metadata", {})
cfg["jokesters_metadata"].update({
    "preset": VRAM_PRESET,
    "source_model": MODEL_ID,
    "quantization": QUANTIZATION,
    "conv_template": CONV_TEMPLATE,
    "notes": PRESET["note"],
})

with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

print("✅ Config written:", config_path)
print(json.dumps({k: cfg.get(k) for k in ["context_window_size", "prefill_chunk_size", "sliding_window_size", "attention_sink_size"]}, indent=2))


## 4. Compile to WebGPU WASM

This is the slow step. TVM bakes your context/prefill sizes into the memory plan.

> ⚠️ **If Colab times out here**, that's normal for 7B/8B. Download the artifact zip in the next section and run locally:
>
> ```bash
> # On a machine with emsdk + mlc-llm built (see scripts/build-vicuna-wasm.sh)
> python -m mlc_llm compile dist/.../mlc-chat-config.json --device webgpu --opt O2 -o my-model.wasm
> ```


In [ ]:
# @title 6️⃣ Compile model_lib (.wasm)
import os, subprocess, sys
from pathlib import Path

ctx = PRESET["context_window_size"]
WASM_NAME = f"{MODEL_SHORT}-{QUANTIZATION}-ctx{ctx}-webgpu.wasm"
WASM_PATH = OUTPUT_DIR / WASM_NAME
CONFIG_PATH = OUTPUT_DIR / "mlc-chat-config.json"

compile_ok = False
compile_error = None

if TRY_COLAB_COMPILE:
    env = os.environ.copy()
    emsdk = Path("/content/emsdk")
    mlc_src = Path("/content/mlc-llm")
    if emsdk.exists():
        # Activate emsdk in subprocess via bash
        cmd = (
            f"source {emsdk}/emsdk_env.sh && "
            f"export TVM_HOME={mlc_src}/3rdparty/tvm && "
            f"export TVM_LIBRARY_PATH={mlc_src}/web/dist/wasm && "
            f"{sys.executable} -m mlc_llm compile {CONFIG_PATH} --device webgpu --opt O2 -o {WASM_PATH}"
        )
        print("🔨 Compiling (this may take 20–60+ minutes)…")
        print(cmd)
        try:
            subprocess.check_call(cmd, shell=True, env=env)
            compile_ok = WASM_PATH.exists()
        except subprocess.CalledProcessError as e:
            compile_error = str(e)
            print("❌ Colab compile failed:", compile_error)
    else:
        compile_error = "Emscripten not found — run toolchain cell first"
        print("❌", compile_error)
else:
    print("⏭️ Skipped compile (TRY_COLAB_COMPILE=False)")

if compile_ok:
    print(f"✅ WASM ready: {WASM_PATH} ({WASM_PATH.stat().st_size / 1e6:.1f} MB)")
elif compile_error:
    print("
📋 Local compile fallback:")
    print(f"  python -m mlc_llm compile {CONFIG_PATH} --device webgpu --opt O2 -o {WASM_NAME}")
    print("  Or: CONTEXT_SIZE={ctx} scripts/build-vicuna-wasm.sh")
else:
    print("No WASM produced yet — use local compile or re-run with GPU/high-RAM runtime")


## 5. Package & download artifacts

Zips everything you need to host on the VPS (`storage.1ink.us`) or HuggingFace.


In [ ]:
# @title 7️⃣ Package artifacts
import json, shutil, zipfile
from pathlib import Path
from IPython.display import FileLink, display

ctx = PRESET["context_window_size"]
bundle_name = f"jokesters-{MODEL_SHORT}-{QUANTIZATION}-ctx{ctx}"
bundle_dir = DIST_ROOT / bundle_name
bundle_dir.mkdir(parents=True, exist_ok=True)

# Copy core artifacts
for pattern in ["mlc-chat-config.json", "tensor-cache.json", "ndarray-cache.json", "tokenizer*", "*.model", "params_shard_*.bin", "*.wasm"]:
    for src in OUTPUT_DIR.glob(pattern):
        shutil.copy2(src, bundle_dir / src.name)

# Integration snippet for models.ts
integration = {
    "model_id": f"jokesters-{MODEL_SHORT}-{QUANTIZATION}-ctx{ctx}",
    "model": f"https://storage.1ink.us/models/{bundle_name}/",
    "model_lib": f"https://storage.1ink.us/models/wasm-libs/{MODEL_SHORT}-{QUANTIZATION}-ctx{ctx}-webgpu.wasm",
    "overrides": {
        "context_window_size": PRESET["context_window_size"],
        "prefill_chunk_size": PRESET["prefill_chunk_size"],
        "sliding_window_size": PRESET.get("sliding_window_size", -1),
        "attention_sink_size": PRESET.get("attention_sink_size", -1),
    },
    "vram_required_MB": 3500 if VRAM_PRESET == "ultra_low" else (5200 if "8B" in MODEL_SHORT else 4000),
}
with open(bundle_dir / "jokesters-models-ts-snippet.json", "w") as f:
    json.dump(integration, f, indent=2)

zip_path = DIST_ROOT / f"{bundle_name}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in bundle_dir.rglob("*"):
        if f.is_file():
            zf.write(f, f.relative_to(bundle_dir.parent))

print(f"📦 Bundle: {bundle_dir}")
print(f"🗜️  Zip:    {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")
print("
Contents:")
!ls -lh {bundle_dir}

display(FileLink(str(zip_path), result_html_prefix="⬇️ Download zip: "))


## 6. Test the new model

Colab's iframe **cannot run WebGPU** reliably. This cell writes a smoke-test HTML you can download and open locally (or use the repo's `public/test-custom-webllm-model.html`).

Serve over HTTP — not `file://` — so model fetches work.


In [ ]:
# @title 8️⃣ Generate local smoke-test HTML
from pathlib import Path
from IPython.display import FileLink, display

ctx = PRESET["context_window_size"]
model_id = f"jokesters-{MODEL_SHORT}-{QUANTIZATION}-ctx{ctx}"

# If WASM was built, point to local path note; user replaces with VPS URL after upload
wasm_file = next(OUTPUT_DIR.glob("*.wasm"), None)
wasm_hint = wasm_file.name if wasm_file else f"{MODEL_SHORT}-{QUANTIZATION}-ctx{ctx}-webgpu.wasm"

html = f'''<!DOCTYPE html>
<html lang="en"><head><meta charset="UTF-8"><title>Jokesters WASM Smoke Test</title></head>
<body style="font-family:system-ui;max-width:800px;margin:2rem auto;padding:0 1rem;background:#111;color:#eee">
<h1>🎭 Custom model smoke test</h1>
<p>Replace <code>MODEL_URL</code> and <code>MODEL_LIB</code> with your hosted URLs after upload.</p>
<pre id="status">Loading WebLLM…</pre>
<button id="go" disabled>Run test prompt</button>
<pre id="out"></pre>
<script type="module">
import * as webllm from "https://esm.run/@mlc-ai/web-llm";
const MODEL_ID = "{model_id}";
const MODEL_URL = "PASTE_WEIGHTS_URL_HERE/";  // e.g. https://storage.1ink.us/models/{model_id}/
const MODEL_LIB = "PASTE_WASM_URL_HERE";       // e.g. https://storage.1ink.us/models/wasm-libs/{wasm_hint}
const OVERRIDES = {json.dumps({k: PRESET[k] for k in ["context_window_size","prefill_chunk_size","sliding_window_size","attention_sink_size"] if k in PRESET})};
const status = document.getElementById('status');
const out = document.getElementById('out');
const go = document.getElementById('go');
let engine;
try {{
  if (!navigator.gpu) throw new Error('WebGPU required (Chrome 113+)');
  const record = {{ model_id: MODEL_ID, model: MODEL_URL, model_lib: MODEL_LIB, overrides: OVERRIDES }};
  engine = await webllm.CreateMLCEngine(MODEL_ID, {{
    initProgressCallback: (r) => status.textContent = r.text,
    appConfig: {{ model_list: [record] }},
  }}, OVERRIDES);
  status.textContent = '✅ Ready';
  go.disabled = false;
}} catch (e) {{ status.textContent = '❌ ' + e.message; }}
go.onclick = async () => {{
  go.disabled = true;
  const stream = await engine.chat.completions.create({{
    messages: [{{role:'user', content:'The Comedian says one punchline about WebGPU:'}}],
    max_tokens: 80, stream: true,
  }});
  let text = '';
  for await (const c of stream) {{
    const d = c.choices?.[0]?.delta?.content; if (d) {{ text += d; out.textContent = text; }}
  }}
  go.disabled = false;
}};
</script></body></html>'''

html_path = DIST_ROOT / f"test-{model_id}.html"
html_path.write_text(html)
print(f"✅ Wrote {html_path}")
print("Also available in the repo: public/test-custom-webllm-model.html")
display(FileLink(str(html_path), result_html_prefix="⬇️ Download test HTML: "))


## 7. Integration with The Jokesters

### Where artifacts go

| Artifact | Destination |
|----------|-------------|
| Weight shards + `mlc-chat-config.json` | VPS: `storage.1ink.us/models/<bundle-name>/` |
| `*.wasm` | VPS: `storage.1ink.us/models/wasm-libs/` |
| Test page | `public/test-custom-webllm-model.html` (already in repo) |

Upload helpers:
- `python scripts/upload_staged_to_vps.py`
- Or HuggingFace `upload_folder` (no credentials in notebooks — use Colab secrets)

### Register in `src/config/models.ts`

Add an entry to `VPS_FP32_MODELS` or `FP16_MODELS` using the generated `jokesters-models-ts-snippet.json`:

```typescript
JOKESTERS_CUSTOM: {
  model_id: "jokesters-vicuna-7b-v1.5-q4f32_1-ctx512",
  model: `${VPS_STORAGE_URL}/jokesters-vicuna-7b-v1.5-q4f32_1-ctx512/`,
  model_lib: `${VPS_STORAGE_URL}/wasm-libs/vicuna-7b-v1.5-q4f32_1-ctx512-webgpu.wasm`,
  overrides: {
    context_window_size: 512,
    prefill_chunk_size: 512,
    sliding_window_size: 256,
    attention_sink_size: 4,
  },
  vram_required_MB: 3500,
}
```

### How this interacts with existing infrastructure

| Component | Role |
|-----------|------|
| `scripts/build-vicuna-wasm.sh` | Local/CI reproducible Vicuna `.wasm` builds (ctx 512/1024) |
| `scripts/build-webllm.sh` | Custom **JS runtime** fork (`3rd_party/web-llm`) — separate from model compile |
| `patches/web-llm/` | Optional runtime patches applied at JS build time |
| `src/llm/MlcEngineAdapter.ts` | Loads models via `CreateMLCEngine` + `appConfig` |
| `src/utils/dynamicContext.ts` | Runtime context trimming + OOM retry |
| `src/service-worker.ts` | Parallel range downloads for shards + `.wasm` |

### Suggested next steps

1. Smoke-test with `public/test-custom-webllm-model.html` (paste VPS URLs).
2. Add the model entry to `src/config/models.ts` and `appConfig.model_list`.
3. Run `npm run dev` — verify load progress + improv scene latency.
4. Compare TTFT with/without custom `.wasm` using browser DevTools Performance tab.
5. For JS-runtime wins (streaming-to-TTS, comedy logit processors), see `docs/webllm-customization-plan.md`.

---

*The Philosopher says: "Optimization without measurement is just anxiety with extra steps."*
*The Comedian says: "But faster tokens *are* the measurement."*
*The Scientist says: "Run the benchmark either way."*


In [ ]:
# @title 9️⃣ (Optional) Upload to HuggingFace
# @markdown Use Colab **Secrets** for your write token — never hardcode tokens in notebooks.
HF_UPLOAD = False  # @param {type:"boolean"}
HF_REPO_ID = "your-username/jokesters-custom-model"  # @param {type:"string"}

if HF_UPLOAD:
    from huggingface_hub import HfApi, login
    from google.colab import userdata
    try:
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = HF_TOKEN
    if not token:
        raise ValueError("Set HF_TOKEN in Colab Secrets or the config cell")
    login(token=token)
    api = HfApi()
    api.create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True)
    api.upload_folder(folder_path=str(bundle_dir), repo_id=HF_REPO_ID, repo_type="model")
    print(f"✅ Uploaded: https://huggingface.co/{HF_REPO_ID}")
else:
    print("⏭️ Skipped HF upload (HF_UPLOAD=False)")
